# Inégalités territoriales d'accès aux équipements sportifs en France

> **Problématique** : Les inégalités territoriales d'accès aux équipements sportifs en France sont-elles le reflet des inégalités socio-économiques ?

**Auteurs** : DELETANG Arthur, GOUALOU Maxence, SERVANT Lucas  
**Cours** : Python pour la Data Science — 2025-2026

---
## Sommaire
1. [Imports](#1---imports)
2. [Collecte des données](#2---collecte-des-données)
3. [Nettoyage et fusion](#3---nettoyage-et-fusion)
4. [Analyse descriptive](#4---analyse-descriptive)
5. [Visualisation cartographique](#5---visualisation-cartographique)
6. [Modélisation](#6---modélisation)
7. [Conclusion](#7---conclusion)

---
## 1 - Imports

In [3]:
!pip install pandas numpy matplotlib seaborn geopandas folium requests beautifulsoup4 scikit-learn lxml

import io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
import folium
from folium.plugins import HeatMap, MarkerCluster
import requests
from bs4 import BeautifulSoup
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 77.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [folium]2m1/4 [lxml]


---
## 2 - Collecte des données

### 2.1 Équipements sportifs — API REST Ministère des Sports

On interroge l'API Opendatasoft du portail Data ES qui expose le recensement national des équipements sportifs (RES). L'export CSV direct évite tout téléchargement manuel.

Source : https://equipements.sports.gouv.fr

In [4]:
url_res = (
    "https://equipements.sports.gouv.fr/api/explore/v2.1/"
    "catalog/datasets/data-es/exports/csv"
    "?lang=fr&timezone=Europe%2FParis&use_labels=true&delimiter=%3B"
)

print("Téléchargement du RES...")
df_res = pd.read_csv(url_res, sep=";", encoding="utf-8", low_memory=False)
print(f"{len(df_res):,} équipements")
df_res.head(3)

Téléchargement du RES...
333,778 équipements


,Numéro de l'équipement sportif,Numéro de l'installation sportive,Date de l'enquête,Nom de l'installation sportive,SIRET Installation,Adresse,Code Postal,Commune nom,Commune INSEE,Type de particularité de l'installation,...,Accessibilité aux personnes en situation de handicap sensoriel aux tribunes,Accessibilité aux personnes en situation de handicap sensoriel aux vestiaires,Longitude,Latitude,Type de particularité de l'installation (brute),Activités,QPV,QPV à 200 mètres,Catégorie,gen_2024Fin_Labellisation
0,E001I171420007,I171420007,2025-03-31,COURT DE TENNIS COUVERT,2.117014e+13,Rue Pierre de Coubertin,17139,Dompierre-sur-Mer,17142,NaN,...,False,False,-1.058679,46.188907,NaN,Tennis,NaN,NaN,NaN,NaN
1,E001I171520001,I171520001,2025-03-31,Court de tennis,2.117015e+13,Rue des Ecoles,17120,Épargnes,17152,NaN,...,False,False,-0.804722,45.541400,NaN,Tennis,NaN,NaN,NaN,NaN
2,E001I171750001,I171750001,2025-03-31,Installations sportives communales,2.117018e+13,6 Route de la grand' maison,17520,Germignac,17175,NaN,...,False,False,-0.336972,45.571800,NaN,Tennis,NaN,NaN,NaN,NaN


### 2.2 Données socio-économiques — INSEE via Data ES

Ce dataset expose les principales variables INSEE au niveau communal : population, superficie, revenu médian (MED20), taux de pauvreté (TP6020), typologie rurale/urbaine. On l'agrège ensuite par département.

Source : https://equipements.sports.gouv.fr/explore/dataset/insee-2020-geoapi-2023

In [5]:
url_insee = (
    "https://equipements.sports.gouv.fr/api/explore/v2.1/catalog/datasets/"
    "insee-2020-geoapi-2023/exports/csv"
    "?lang=fr&timezone=Europe%2FParis&use_labels=true&delimiter=%3B"
)

print("Téléchargement données INSEE communes...")
df_communes = pd.read_csv(url_insee, sep=";", encoding="utf-8", low_memory=False)
print(f"{len(df_communes):,} communes — {df_communes.shape[1]} colonnes")
df_communes.head(3)

Téléchargement données INSEE communes...
35,075 communes — 47 colonnes


,codeCommune,nomCommune,codeEpci,nomEpci,codeDepartement,nomDepartement,codeRegion,nomRegion,population,surface,...,lib_ville,code_bdv,lib_bdv,zfrr,zrr,TYPO_RURB_CRTE,vas,dens_niveau,dens_lib,commune_loi_montagne
0,84118,Saint-Saturnin-lès-Apt,200040624,CC Pays d'Apt-Luberon,84,Vaucluse,93,Provence-Alpes-Côte d'Azur,2910,7682.46,...,Saint-Saturnin-lès-Apt,84003,Apt,zfrr,NaN,RURAL,NaN,6.0,Rural à habitat dispersé,Loi Montagne
1,84119,Saint-Saturnin-lès-Avignon,248400251,CA du Grand Avignon (COGA),84,Vaucluse,93,Provence-Alpes-Côte d'Azur,5027,621.81,...,Saint-Saturnin-lès-Avignon,84007,Avignon,NaN,NaN,URBAIN,NaN,4.0,Ceintures urbaines,NaN
2,84120,Saint-Trinit,200035723,CC Ventoux Sud,84,Vaucluse,93,Provence-Alpes-Côte d'Azur,120,1690.17,...,Saint-Trinit,84123,Sault,zfrr,NaN,RURAL,NaN,7.0,Rural à habitat très dispersé,Loi Montagne


---
## 3 - Nettoyage et fusion

Cette section explore d'abord la structure brute des deux sources principales, identifie les problèmes (valeurs manquantes, codes aberrants, doublons), les corrige, puis construit le DataFrame analytique final au niveau département.

### 3.1 Exploration et nettoyage du RES

In [6]:
# Structure générale du RES
print(f"Dimensions : {df_res.shape}")
print(f"\nColonnes disponibles :")
print(df_res.columns.tolist())

Dimensions : (333778, 113)

Colonnes disponibles :
["Numéro de l'équipement sportif", "Numéro de l'installation sportive", "Date de l'enquête", "Nom de l'installation sportive", 'SIRET Installation', 'Adresse', 'Code Postal', 'Commune nom', 'Commune INSEE', "Type de particularité de l'installation", 'Unité Administrative Immatriculée (UAI)', "Accessibilité de l'installation en fonction du type handicap", "Accessibilité de l'installation en transport en commun des différents mode", 'Observation Installation', "Date de changement d'état de la fiche d'enquête", "Date de création de la fiche d'enquête", 'Installation hors-service', 'EPCI INSEE', 'EPCI Nom', 'Département Code', 'Département Nom', 'Région Code', 'Région Nom', 'Bassin de vie Code', 'Bassin de vie Nom', 'Arrondissement Code', 'Arrondissement Nom', 'Densite Niveau', 'Densite Catégorie', 'Département Code Complet', 'Réctorat Nom', 'ZRR Simplifié', "Nom de l'équipement sportif", "Type d'équipement sportif", 'Coordonnées', 'Nom du

In [8]:
# Valeurs manquantes dans les colonnes clés du RES
cols_cles_res = [
    "Département Code", "Commune nom",
    "Type d'équipement sportif", "Famille d'équipement sportif"
]
na_res = df_res[cols_cles_res].isnull().sum().rename("NaN")
na_res_pct = (df_res[cols_cles_res].isnull().mean() * 100).rename("%")
print("Valeurs manquantes dans les colonnes clés du RES :")
print(pd.concat([na_res, na_res_pct.round(1)], axis=1))

Valeurs manquantes dans les colonnes clés du RES :
                              NaN    %
Département Code              199  0.1
Commune nom                     0  0.0
Type d'équipement sportif       1  0.0
Famille d'équipement sportif    1  0.0


In [9]:
# Vérification des codes département dans le RES
codes_dep_res = df_res["Département Code"].astype(str).str.strip().unique()
codes_dep_res_sorted = sorted(codes_dep_res)
print(f"{len(codes_dep_res_sorted)} codes départements uniques dans le RES :")
print(codes_dep_res_sorted)

108 codes départements uniques dans le RES :
['1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '21', '22', '23', '24', '25', '26', '27', '28', '29', '2A', '2B', '3', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '4', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '5', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '6', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '7', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '8', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '9', '90', '91', '92', '93', '94', '95', '971', '972', '973', '974', '975', '976', '977', '978', '986', '987', '988', 'nan']


In [10]:
# Vérification des doublons sur le numéro d'équipement
n_doublons = df_res.duplicated(subset=["Numéro de l'équipement sportif"]).sum()
print(f"Doublons sur le numéro d'équipement : {n_doublons}")

Doublons sur le numéro d'équipement : 84


In [11]:
# Nettoyage du RES :
# - standardiser le code département sur 2 caractères
# - supprimer les lignes sans code département valide
df_res["code_dep"] = df_res["Département Code"].astype(str).str.strip().str.zfill(2)

# Garder uniquement les départements valides (métropole + DOM)
mask_dep_valide = df_res["code_dep"].str.match(r"^\d{2}$|^2[AB]$")
n_exclus = (~mask_dep_valide).sum()
df_res = df_res[mask_dep_valide].copy()

print(f"{n_exclus} lignes exclues (code département invalide)")
print(f"RES nettoyé : {len(df_res):,} équipements")

9484 lignes exclues (code département invalide)
RES nettoyé : 324,294 équipements
